# Sanity check — EFC_compositional

Run this **before any analysis**, on every subject, as soon as a session is collected.

It answers two questions:

1. **Does every trial in the `.dat` match its target file?** (chordID and all timing parameters)
2. **Does any block have extra rows?** This is the common failure mode — a run stopped
   partway through because of a mistake and then redone, which leaves the aborted attempt
   in the `.dat` on top of the good one.

Anything wrong is printed with the offending rows. Set `SUBJECT` below and run all cells.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------- configuration
SUBJECT = 101          # subject number to check
MOV_CHECK = True       # also verify one .mov per collected block

# project root = the folder that contains target/ and data/
ROOT = Path.cwd()
while not (ROOT / "target").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.design import day_blocks

# Session structure: which block numbers belong to each day, from utils/design.py
# (s100 stops at block 38, everyone else runs to 52). Every filename is built from
# this, so a missing or misnamed file is reported as an error rather than quietly
# dropping out of the checks the way a glob would let it.
DAY_BLOCKS = day_blocks(SUBJECT)
DAT_PATH = ROOT / "data" / f"s{SUBJECT}" / f"EFC_compositional_{SUBJECT}.dat"
TGT_DIR = ROOT / "target"
MOV_DIR = DAT_PATH.parent

TGT_NAME = "s{sn}_day{day}_{bn:02d}.tgt"
MOV_NAME = "EFC_compositional_{sn}_{bn:02d}.mov"

# columns that must be identical between the .dat and the .tgt, trial by trial
MATCH_COLS = ["subNum", "chordID", "planTime", "success_holdTime",
              "execMaxTime", "feedbackTime", "iti"]

EXPECTED_BLOCKS = sorted(b for blocks in DAY_BLOCKS.values() for b in blocks)
DAY_OF_BLOCK = {b: d for d, blocks in DAY_BLOCKS.items() for b in blocks}

print("root   :", ROOT)
print("dat    :", DAT_PATH.relative_to(ROOT), "\u2014", "found" if DAT_PATH.exists() else "MISSING")
print("blocks :", f"{EXPECTED_BLOCKS[0]}-{EXPECTED_BLOCKS[-1]}",
      "across days", sorted(DAY_BLOCKS))

root   : /Users/aghavamp/Desktop/Projects/EFC_compositional
dat    : data/s101/EFC_compositional_101.dat — found
blocks : 1-52 across days [1, 2, 3, 4]


In [2]:
# ------------------------------------------------------------- reporting helpers
FAILURES, WARNINGS = [], []

def section(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

def ok(msg):
    print(f"  [ OK ]   {msg}")

def fail(msg, detail=None):
    FAILURES.append(msg)
    print(f"  [FAIL]   {msg}")
    if detail is not None:
        text = detail.to_string() if isinstance(detail, (pd.DataFrame, pd.Series)) else str(detail)
        print("\n".join("           " + ln for ln in text.splitlines()))

def warn(msg, detail=None):
    WARNINGS.append(msg)
    print(f"  [WARN]   {msg}")
    if detail is not None:
        text = detail.to_string() if isinstance(detail, (pd.DataFrame, pd.Series)) else str(detail)
        print("\n".join("           " + ln for ln in text.splitlines()))

In [3]:
# ------------------------------------------------------------------- load data
if not DAT_PATH.exists():
    raise FileNotFoundError(f"No .dat for subject {SUBJECT} at {DAT_PATH}")

dat = pd.read_csv(DAT_PATH, sep="\t")
dat = dat.loc[:, ~dat.columns.str.startswith("Unnamed")]   # trailing tab on every row
dat.columns = dat.columns.str.strip()
dat["row"] = np.arange(2, len(dat) + 2)                     # line number in the file
dat["DateTime"] = pd.to_datetime(dat["DateTime"], format="%d-%m-%Y,%H:%M:%S", errors="coerce")

# ------------------------------------------------------------------ load targets
targets, missing_targets = {}, []
for bn in EXPECTED_BLOCKS:
    f = TGT_DIR / TGT_NAME.format(sn=SUBJECT, day=DAY_OF_BLOCK[bn], bn=bn)
    if not f.exists():
        missing_targets.append(f.name)
        continue
    t = pd.read_csv(f, sep="\t")
    t.columns = t.columns.str.strip()
    targets[bn] = dict(file=f.name, day=DAY_OF_BLOCK[bn], trials=t)

print(f"{len(dat)} rows in the .dat, "
      f"{len(targets)} of {len(EXPECTED_BLOCKS)} expected target files for subject {SUBJECT}")
dat.head(3)

840 rows in the .dat, 52 of 52 expected target files for subject 101


,BN,TN,subNum,chordID,planTime,max_holdTime,success_holdTime,execMaxTime,feedbackTime,iti,...,extTopThresh,extBotThresh,flexTopThresh,flexBotThresh,RT,ET,trialPoint,planError,DateTime,row
0,1,1,101,12921,1364,600,600,10000,1000,500,...,5,2,-5,-2,666.0,2306.0,1,0,2026-09-21 13:03:00,2
1,1,2,101,12921,1228,600,600,10000,1000,500,...,5,2,-5,-2,426.0,1060.0,1,0,2026-09-21 13:03:08,3
2,1,3,101,12921,1279,600,600,10000,1000,500,...,5,2,-5,-2,404.0,714.0,1,0,2026-09-21 13:03:13,4


## Check 1 — block inventory

Which blocks exist in the data, which targets they correspond to, and whether anything
is orphaned on either side.

In [4]:
section("CHECK 1 - block inventory")

dat_blocks = sorted(int(b) for b in dat["BN"].unique())
tgt_blocks = sorted(targets)

no_target = sorted(set(dat_blocks) - set(tgt_blocks))
not_run = sorted(set(tgt_blocks) - set(dat_blocks))

print(f"  expected blocks   : {EXPECTED_BLOCKS[0]}-{EXPECTED_BLOCKS[-1]} ({len(EXPECTED_BLOCKS)})")
print(f"  blocks in .dat    : {dat_blocks}")
print(f"  target files found: {tgt_blocks}\n")

if missing_targets:
    fail(f"{len(missing_targets)} expected target file(s) missing or misnamed",
         pd.Series(missing_targets, name="expected name"))
else:
    ok(f"all {len(EXPECTED_BLOCKS)} expected target files present and correctly named")

if no_target:
    fail(f"data blocks with NO target file: {no_target}")
else:
    ok("every block in the .dat has a matching target file")

unexpected = sorted(set(dat_blocks) - set(EXPECTED_BLOCKS))
if unexpected:
    fail(f"blocks in the .dat outside the session structure: {unexpected}")

if not_run:
    warn(f"targets not yet collected (expected if the study is in progress): {not_run}")
else:
    ok("every target file has data")


CHECK 1 - block inventory
  expected blocks   : 1-52 (52)
  blocks in .dat    : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]
  target files found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]

  [ OK ]   all 52 expected target files present and correctly named
  [ OK ]   every block in the .dat has a matching target file
  [WARN]   targets not yet collected (expected if the study is in progress): [43, 44, 45, 46, 47, 48, 49, 50, 51, 52]


## Check 2 — extra rows and restarted runs 

**The important one.** A run stopped partway and redone leaves the aborted attempt in the
`.dat` in front of the good one, so the block ends up with more than its expected number
of rows and the `TN` column restarts partway through.

Each block is split into *attempts* at every point where `TN` stops increasing. A clean
block has exactly one attempt, running `1..n`.

In [5]:
section("CHECK 2 - extra rows / restarted runs")

def split_attempts(block):
    """Split one block's rows (file order) into run attempts at each TN reset."""
    attempts, current, prev_tn = [], [], None
    for idx, tn in zip(block.index, block["TN"]):
        if prev_tn is not None and tn <= prev_tn:
            attempts.append(current)
            current = []
        current.append(idx)
        prev_tn = tn
    if current:
        attempts.append(current)
    return attempts

drop_idx = []        # rows belonging to aborted attempts
block_report = []

for bn in dat_blocks:
    block = dat[dat["BN"] == bn]
    n_exp = len(targets[bn]["trials"]) if bn in targets else np.nan
    attempts = split_attempts(block)

    if len(attempts) > 1:
        keep = None
        for a in attempts:                      # prefer the last attempt of full length
            if len(a) == n_exp:
                keep = a
        if keep is None:
            keep = max(attempts, key=len)
        detail = pd.DataFrame([{
            "attempt": i + 1,
            "rows_in_file": f"{dat.loc[a, 'row'].min()}-{dat.loc[a, 'row'].max()}",
            "n_trials": len(a),
            "TN": f"{dat.loc[a, 'TN'].min()}-{dat.loc[a, 'TN'].max()}",
            "started": dat.loc[a, "DateTime"].min(),
            "verdict": "KEEP (complete)" if a is keep else "DROP (aborted)",
        } for i, a in enumerate(attempts)])
        fail(f"block {bn}: RESTARTED - {len(attempts)} attempts, "
             f"{len(block)} rows for {n_exp} expected trials", detail)
        for a in attempts:
            if a is not keep:
                drop_idx.extend(a)

    elif not np.isnan(n_exp) and len(block) > n_exp:
        fail(f"block {bn}: {len(block)} rows but target has {n_exp} trials "
             f"(extra rows, TN did not reset)", block[["row", "BN", "TN", "chordID"]].tail(8))

    elif not np.isnan(n_exp) and len(block) < n_exp:
        warn(f"block {bn}: INCOMPLETE - {len(block)} rows, target has {n_exp} trials "
             f"(run aborted and never redone?)")

    else:
        expected_tn = list(range(1, len(block) + 1))
        if list(block["TN"]) != expected_tn:
            fail(f"block {bn}: TN is not 1..{len(block)}", block[["row", "BN", "TN"]])

    block_report.append(dict(BN=bn, n_rows=len(block), n_expected=n_exp,
                             n_attempts=len(attempts)))

if not FAILURES:
    ok(f"all {len(dat_blocks)} blocks have exactly one complete attempt with TN = 1..n")

pd.DataFrame(block_report).set_index("BN").T


CHECK 2 - extra rows / restarted runs
  [ OK ]   all 42 blocks have exactly one complete attempt with TN = 1..n


BN,1,2,3,4,5,6,7,8,9,10,...,33,34,35,36,37,38,39,40,41,42
n_rows,20,20,20,20,20,20,20,20,20,20,...,20,20,20,20,20,20,20,20,20,20
n_expected,20,20,20,20,20,20,20,20,20,20,...,20,20,20,20,20,20,20,20,20,20
n_attempts,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [6]:
# `clean` excludes aborted attempts; every later check runs on it.
clean = dat.drop(index=drop_idx).reset_index(drop=True)
if drop_idx:
    print(f"{len(drop_idx)} row(s) from aborted attempts excluded -> "
          f"{len(clean)} of {len(dat)} rows retained")
    print("See the last cell to write a cleaned .dat.")
else:
    print(f"No aborted attempts. Using all {len(clean)} rows.")

No aborted attempts. Using all 840 rows.


## Check 3 — per-trial match against the target files 

For every trial: does the `chordID` and every timing parameter in the `.dat` equal the
value in the corresponding row of the `.tgt`? Any mismatch means the wrong target file was
run for that block, or the files have drifted out of sync.

In [7]:
section("CHECK 3 - per-trial match against targets")

mismatches = []
for bn in sorted(int(b) for b in clean["BN"].unique()):
    if bn not in targets:
        continue
    d = clean[clean["BN"] == bn].reset_index(drop=True)
    t = targets[bn]["trials"].reset_index(drop=True)
    n = min(len(d), len(t))
    for col in MATCH_COLS:
        if col not in d.columns or col not in t.columns:
            continue
        dv, tv = d[col].iloc[:n], t[col].iloc[:n]
        bad = np.where(dv.values != tv.values)[0]
        for i in bad:
            mismatches.append(dict(BN=bn, TN=int(d["TN"].iloc[i]),
                                   file_row=int(d["row"].iloc[i]), column=col,
                                   dat=dv.iloc[i], target=tv.iloc[i],
                                   target_file=targets[bn]["file"]))

n_checked = sum(len(clean[clean["BN"] == b]) for b in clean["BN"].unique() if b in targets)
if mismatches:
    mm = pd.DataFrame(mismatches)
    fail(f"{len(mm)} trial/target mismatches across {mm['BN'].nunique()} block(s)", mm.head(40))
    if len(mm) > 40:
        print(f"           ... {len(mm) - 40} more")
else:
    ok(f"all {n_checked} trials match their target on {', '.join(MATCH_COLS)}")


CHECK 3 - per-trial match against targets
  [ OK ]   all 840 trials match their target on subNum, chordID, planTime, success_holdTime, execMaxTime, feedbackTime, iti


## Check 4 — file structure and chronology

Duplicated rows, unparsable timestamps, and blocks recorded out of order. A block that
starts before the previous one ended points at an editing or concatenation mistake.

In [8]:
section("CHECK 4 - structure and chronology")

dup = clean[clean.duplicated(subset=["BN", "TN"], keep=False)]
if len(dup):
    fail(f"{len(dup)} rows share a (BN, TN) pair", dup[["row", "BN", "TN", "chordID"]])
else:
    ok("no duplicated (BN, TN) pairs")

if clean["DateTime"].isna().any():
    fail(f"{int(clean['DateTime'].isna().sum())} rows have an unparsable DateTime")
else:
    ok("all DateTime values parsed")

    out_of_order, prev_end = [], None
    for bn in sorted(int(b) for b in clean["BN"].unique()):
        ts = clean.loc[clean["BN"] == bn, "DateTime"]
        if not ts.is_monotonic_increasing:
            out_of_order.append(f"block {bn}: timestamps not increasing within the block")
        if prev_end is not None and ts.iloc[0] < prev_end:
            out_of_order.append(f"block {bn}: starts before block {bn - 1} ended")
        prev_end = ts.iloc[-1]
    if out_of_order:
        fail("blocks are out of chronological order", pd.Series(out_of_order))
    else:
        ok("blocks run in chronological order")

    # the day in the target filename should agree with the calendar date of the recording
    bad_day = []
    for bn in sorted(int(b) for b in clean["BN"].unique()):
        if bn not in targets:
            continue
        dates = clean.loc[clean["BN"] == bn, "DateTime"].dt.date.unique()
        if len(dates) > 1:
            bad_day.append(f"block {bn}: spans {len(dates)} calendar dates {list(dates)}")
    day_map = (clean.assign(day=clean["BN"].map(lambda b: targets[b]["day"] if b in targets else np.nan))
                    .groupby("day")["DateTime"].agg(lambda s: sorted(s.dt.date.unique())))
    for day, dates in day_map.items():
        if len(dates) > 1:
            bad_day.append(f"day {int(day)} target files span {len(dates)} dates {dates}")
    if bad_day:
        warn("day labels and recording dates disagree", pd.Series(bad_day))
    else:
        ok("each session day maps to a single calendar date")
    print()
    print(day_map.to_string())


CHECK 4 - structure and chronology
  [ OK ]   no duplicated (BN, TN) pairs
  [ OK ]   all DateTime values parsed
  [ OK ]   blocks run in chronological order
  [ OK ]   each session day maps to a single calendar date

day
1    [2026-09-21]
2    [2026-09-23]
3    [2026-09-25]


## Check 5 — mov files

One `.mov` per collected block. Mean deviation comes from these force traces, so a missing
or truncated file costs that block's kinematics even when the `.dat` looks fine.

In [9]:
section("CHECK 5 - .mov files")

if not MOV_CHECK:
    print("  skipped (MOV_CHECK = False)")
else:
    collected = sorted(int(b) for b in clean["BN"].unique())
    movs = {}
    for bn in EXPECTED_BLOCKS:
        f = MOV_DIR / MOV_NAME.format(sn=SUBJECT, bn=bn)
        if f.exists():
            movs[bn] = f

    missing = [b for b in collected if b not in movs]
    orphan = sorted(set(movs) - set(collected))
    tiny = {b: f.stat().st_size for b, f in movs.items() if f.stat().st_size < 1_000_000}

    if missing:
        fail(f"blocks with data but no .mov: {missing}")
    else:
        ok(f"all {len(collected)} collected blocks have a correctly named .mov")
    if orphan:
        warn(f".mov files with no matching data block: {orphan}")
    if tiny:
        warn("suspiciously small .mov files (<1 MB)", pd.Series(tiny, name="bytes"))
    elif movs:
        sizes = pd.Series({b: f.stat().st_size / 1e6 for b, f in movs.items()})
        ok(f".mov sizes {sizes.min():.1f}-{sizes.max():.1f} MB, median {sizes.median():.1f} MB")


CHECK 5 - .mov files
  [ OK ]   all 42 collected blocks have a correctly named .mov
  [ OK ]   .mov sizes 7.5-13.5 MB, median 8.0 MB


## Check 6 — value sanity

Ranges and flags. `RT == 0` means the subject never left the baseline zone and `ET == 0`
means the chord was never acquired — both are "event never happened" sentinels, not real
measurements, so they must never appear on a correct trial. **Drop them before averaging
RT/ET rather than treating them as zero.**

In [10]:
section("CHECK 6 - value sanity")

for col, lo, hi in [("trialPoint", 0, 1), ("planError", 0, 1)]:
    bad = clean[~clean[col].isin([lo, hi])]
    if len(bad):
        fail(f"{col} has values outside {{{lo},{hi}}}: {sorted(bad[col].unique())}")
    else:
        ok(f"{col} is always {lo}/{hi}")

# a plan error can never be a rewarded trial
bad = clean[(clean["planError"] == 1) & (clean["trialPoint"] == 1)]
if len(bad):
    fail(f"{len(bad)} trials flagged planError but scored as correct", bad[["row", "BN", "TN"]])
else:
    ok("no trial is both a plan error and correct")

# sentinels must not appear on correct trials
for col in ("RT", "ET"):
    bad = clean[(clean[col] == 0) & (clean["trialPoint"] == 1)]
    if len(bad):
        fail(f"{len(bad)} CORRECT trials have {col} == 0 (sentinel value)",
             bad[["row", "BN", "TN", "RT", "ET"]])
    else:
        n = int(((clean[col] == 0) & (clean["trialPoint"] == 0)).sum())
        ok(f"{col} == 0 only on failed trials ({n} of them) - exclude these from {col} means")

neg = clean[(clean["RT"] < 0) | (clean["ET"] < 0)]
if len(neg):
    fail(f"{len(neg)} rows have a negative RT or ET", neg[["row", "BN", "TN", "RT", "ET"]])
else:
    ok("no negative RT or ET")

over = clean[clean["ET"] > clean["execMaxTime"]]
if len(over):
    fail(f"{len(over)} rows have ET greater than execMaxTime", over[["row", "BN", "TN", "ET", "execMaxTime"]])
else:
    ok("ET never exceeds execMaxTime")

print()
print(clean[["RT", "ET"]].describe().loc[["min", "50%", "max"]].to_string())
print()
print(f"  success rate overall : {100 * clean['trialPoint'].mean():.1f}%")
print(f"  plan errors          : {int(clean['planError'].sum())} of {len(clean)} "
      f"({100 * clean['planError'].mean():.1f}%)")


CHECK 6 - value sanity
  [ OK ]   trialPoint is always 0/1
  [ OK ]   planError is always 0/1
  [ OK ]   no trial is both a plan error and correct
  [ OK ]   RT == 0 only on failed trials (0 of them) - exclude these from RT means
  [ OK ]   ET == 0 only on failed trials (2 of them) - exclude these from ET means
  [ OK ]   no negative RT or ET
  [ OK ]   ET never exceeds execMaxTime

        RT      ET
min    2.0     0.0
50%  346.0   377.0
max  884.0  9654.0

  success rate overall : 97.7%
  plan errors          : 8 of 840 (1.0%)


## Summary

In [11]:
section(f"SUMMARY - subject {SUBJECT}")

print(f"  rows in .dat        : {len(dat)}")
print(f"  rows after cleaning : {len(clean)}")
print(f"  blocks collected    : {len(clean['BN'].unique())}")
print(f"  failures            : {len(FAILURES)}")
print(f"  warnings            : {len(WARNINGS)}\n")

if FAILURES:
    print("  FAILURES - fix these before analysing:")
    for f in FAILURES:
        print(f"    - {f}")
if WARNINGS:
    print("\n  Warnings - check these are expected:")
    for w in WARNINGS:
        print(f"    - {w}")

print()
if not FAILURES:
    print(f"  *** PASS - subject {SUBJECT} is safe to analyse ***")
else:
    print(f"  *** {len(FAILURES)} PROBLEM(S) FOUND - see above ***")


SUMMARY - subject 101
  rows in .dat        : 840
  rows after cleaning : 840
  blocks collected    : 42
  failures            : 0
  warnings            : 1


  Warnings - check these are expected:
    - targets not yet collected (expected if the study is in progress): [43, 44, 45, 46, 47, 48, 49, 50, 51, 52]

  *** PASS - subject 101 is safe to analyse ***
